# Laboratorio 01 — Exploración libre con PySpark

**Semana:** 02  
**Actividad de referencia:** Actividad 01  
**Estudiante:** Daniel Guzmán  
**Dataset:** Cards Data — Financial Transaction Dataset  

## Parte 1 — Descripción del dataset

### Nombre y fuente

El dataset seleccionado es `cards_data.csv`, parte del conjunto **Financial Transaction Dataset** usado durante la Semana 02 del bootcamp.  
La fuente fue el SharePoint del bootcamp:

`inetum_data_engineer_bootcamp / semana_02 / financial_transaction_dataset`

### Dominio

El dominio del dataset es **financiero/bancario**. Describe información de tarjetas asociadas a clientes, incluyendo tipo de tarjeta, marca, límite de crédito, presencia de chip y fechas relevantes.

### ¿Por qué elegí este dataset?

Elegí este archivo porque permite analizar características de tarjetas bancarias y entender posibles patrones de negocio relacionados con tipo de tarjeta, marca, límites de crédito, tecnología chip y antigüedad de las tarjetas.

### Número aproximado de filas y archivos

Este laboratorio trabaja con un solo archivo:

- `cards_data.csv`

El número exacto de filas será calculado con PySpark en la carga del DataFrame.

### Preguntas de negocio

1. ¿Qué tipo de tarjeta es más frecuente en el dataset?
2. ¿Qué marca de tarjeta tiene mayor límite de crédito promedio?
3. ¿Las tarjetas con chip tienen límites de crédito diferentes frente a las tarjetas sin chip?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan, count as spark_count, sum as spark_sum

MI_NOMBRE = "daniel"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"
ARCHIVO = "cards_data.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/{ARCHIVO}")

print(f"Filas: {df.count():,}")
print(f"Columnas: {len(df.columns)}")

In [0]:
# Schema — tipos de cada columna
df.printSchema()

## Observaciones del schema

El dataset tiene **6,146 filas** y **13 columnas**.

`inferSchema` interpretó varias columnas como numéricas, por ejemplo `id`, `client_id`, `card_number`, `cvv` y `num_cards_issued`.

Algunas columnas financieras como `credit_limit` pueden requerir limpieza adicional, porque normalmente vienen como texto con símbolo de moneda o separadores. También hay columnas de fecha como `expires` y `acct_open_date`, que pueden necesitar conversión a tipo fecha para análisis temporal.

In [0]:
# Vista rápida de los primeros registros
df.show(10, truncate=False)

In [0]:
# Estadísticas descriptivas de columnas numéricas y string
df.describe().show(truncate=False)

## Observaciones del describe

El `describe()` permite revisar conteos, mínimos, máximos, promedios y desviaciones estándar.

En este dataset es importante revisar especialmente columnas como `credit_limit`, `num_cards_issued`, `year_pin_last_changed` y `card_on_dark_web`.

También se debe tener cuidado con columnas que parecen numéricas pero representan identificadores, como `card_number` o `cvv`, porque no deberían analizarse como métricas de negocio.

In [0]:
# Análisis de nulos y vacíos por columna

total = df.count()
numeric_types = {"double", "float", "long", "integer", "short", "byte"}

perfil = df.select([
    spark_sum(
        when(
            col(c).isNull() |
            (isnan(col(c)) if df.schema[c].dataType.typeName() in numeric_types else F.lit(False)) |
            (col(c).cast("string") == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

nulos = perfil.collect()[0].asDict()

print(f"{'Columna':<35} {'Nulos':>8} {'% Nulos':>10}")
print("-" * 56)

for c, n in sorted(nulos.items(), key=lambda x: -x[1]):
    print(f"{c:<35} {n:>8,} {n/total*100:>9.1f}%")

## Observaciones de nulos

Se revisaron valores nulos y vacíos en todas las columnas del dataset.

En un pipeline real, las columnas críticas para análisis financiero, como `card_type`, `card_brand`, `credit_limit` o `has_chip`, deberían validarse cuidadosamente antes de usarse en reportes.

Si una columna tuviera pocos nulos, se podría imputar o mantener según el caso. Si tuviera demasiados nulos, habría que revisar si realmente aporta valor o si representa un problema de calidad en la fuente.

In [0]:
# Cardinalidad de columnas categóricas tipo string

string_cols = [f.name for f in df.schema.fields if f.dataType.typeName() == "string"]

print(f"{'Columna':<35} {'Valores únicos':>15}")
print("-" * 52)

for c in string_cols:
    card = df.select(c).distinct().count()
    print(f"{c:<35} {card:>15,}")

## Observaciones de cardinalidad

Las columnas con baja cardinalidad son buenas candidatas para análisis con `groupBy`, por ejemplo `card_brand`, `card_type`, `has_chip` o `card_on_dark_web`.

Las columnas con alta cardinalidad o valores únicos, como números de tarjeta, identificadores o fechas específicas, no suelen ser buenas dimensiones para agrupaciones generales.

Este análisis ayuda a decidir qué columnas pueden servir para segmentar el negocio.

In [0]:
# Transformación 1 — Normalizar nombres de columnas a snake_case

import re

def to_snake_case(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^a-z0-9]+", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name).strip("_")
    return col_name

df_clean = df

for old_col in df.columns:
    df_clean = df_clean.withColumnRenamed(old_col, to_snake_case(old_col))

print(df_clean.columns)

## Explicación transformación 1

Se normalizaron los nombres de columnas a formato `snake_case`.

Esto ayuda a trabajar de forma más cómoda en PySpark, evita problemas con espacios o caracteres especiales y deja el dataset más preparado para una capa Silver en una arquitectura Medallón.

In [0]:
# Transformación 2 — Limpiar credit_limit y convertirlo a double

df_clean = df_clean.withColumn(
    "credit_limit_num",
    F.regexp_replace(F.col("credit_limit"), "[$,]", "").cast("double")
)

df_clean.select("credit_limit", "credit_limit_num").show(10, truncate=False)

## Explicación transformación 2

La columna `credit_limit` representa un valor financiero, pero puede venir como texto con símbolo de moneda o separadores.

Se creó `credit_limit_num` como columna numérica para poder calcular promedios, máximos, mínimos y comparaciones por tipo o marca de tarjeta.

In [0]:
# Transformación 3 — Convertir columnas de fecha

df_clean = (
    df_clean
    .withColumn("expires_date", F.to_date(F.col("expires"), "MM/yyyy"))
    .withColumn("acct_open_date_parsed", F.to_date(F.col("acct_open_date"), "MM/yyyy"))
    .withColumn("anio_apertura", F.year(F.col("acct_open_date_parsed")))
    .withColumn("anio_expiracion", F.year(F.col("expires_date")))
)

df_clean.select(
    "expires",
    "expires_date",
    "acct_open_date",
    "acct_open_date_parsed",
    "anio_apertura",
    "anio_expiracion"
).show(10, truncate=False)

## Explicación transformación 3

Se convirtieron las columnas `expires` y `acct_open_date` a tipo fecha.

Esto permite extraer componentes temporales como año de apertura y año de expiración, útiles para analizar antigüedad de tarjetas o distribución temporal de emisión.

In [0]:
# Transformación 4 — Crear columna de segmento por límite de crédito y filtrar tarjetas de límite alto

df_clean = df_clean.withColumn(
    "segmento_limite",
    F.when(F.col("credit_limit_num") >= 20000, "alto")
     .when(F.col("credit_limit_num") >= 10000, "medio")
     .otherwise("bajo")
)

df_limite_alto = df_clean.filter(F.col("segmento_limite") == "alto")

print(f"Tarjetas con límite alto: {df_limite_alto.count():,}")

df_limite_alto.select(
    "id",
    "client_id",
    "card_brand",
    "card_type",
    "credit_limit_num",
    "segmento_limite"
).show(10, truncate=False)

## Explicación transformación 4

Se creó la columna `segmento_limite` para clasificar las tarjetas según su límite de crédito.

Esta transformación permite analizar el portafolio de tarjetas por segmentos de riesgo o valor. También se filtraron las tarjetas de límite alto para revisar qué características tienen.

In [0]:
# Parte 4 — Evaluación lazy y plan de ejecución
# Cadena de transformaciones: filter + withColumn + groupBy/agg
# Hasta antes de explain/show, Spark solo construye el plan lógico.

df_plan = (
    df_clean
    .filter(F.col("credit_limit_num").isNotNull())
    .withColumn(
        "tiene_chip",
        F.when(F.col("has_chip") == "YES", True).otherwise(False)
    )
    .groupBy("card_brand", "card_type", "tiene_chip")
    .agg(
        F.count("*").alias("total_tarjetas"),
        F.round(F.avg("credit_limit_num"), 2).alias("limite_promedio"),
        F.max("credit_limit_num").alias("limite_maximo")
    )
    .orderBy(F.col("limite_promedio").desc())
)

df_plan.explain()


## Evaluación lazy y plan de ejecución

En esta cadena se aplicaron varias transformaciones: un filtro, una columna derivada, una agrupación, agregaciones y un ordenamiento.

Spark no ejecuta inmediatamente cada transformación. Primero construye un plan lógico y físico de ejecución. En el plan se puede observar que Spark prepara las operaciones que realizará cuando se invoque una acción como `show()`, `count()` o `write()`.

El `groupBy` y el `orderBy` pueden generar movimientos de datos entre particiones, conocidos como shuffle, porque Spark necesita reorganizar los datos para agrupar y ordenar correctamente.


In [0]:
# Acción: aquí Spark materializa el plan
df_plan.show(10, truncate=False)

In [0]:
# Pregunta 1: ¿Qué tipo de tarjeta es más frecuente?

df_tipo_tarjeta = (
    df_clean
    .groupBy("card_type")
    .agg(
        F.count("*").alias("total_tarjetas"),
        F.round(F.count("*") / df_clean.count() * 100, 2).alias("porcentaje")
    )
    .orderBy(F.col("total_tarjetas").desc())
)

df_tipo_tarjeta.show(truncate=False)

## Conclusión pregunta 1

El tipo de tarjeta más frecuente en el dataset es **Debit**, con **3,511 tarjetas**, equivalente al **57.13%** del total.

En segundo lugar aparecen las tarjetas **Credit**, con **2,057 tarjetas** (**33.47%**), y finalmente **Debit (Prepaid)**, con **578 tarjetas** (**9.40%**).

Esto muestra que el portafolio está concentrado principalmente en tarjetas débito.

In [0]:
# Pregunta 2: ¿Qué marca de tarjeta tiene mayor límite de crédito promedio?

df_limite_por_marca = (
    df_clean
    .groupBy("card_brand")
    .agg(
        F.count("*").alias("total_tarjetas"),
        F.round(F.avg("credit_limit_num"), 2).alias("limite_promedio"),
        F.max("credit_limit_num").alias("limite_maximo")
    )
    .orderBy(F.col("limite_promedio").desc())
)

df_limite_por_marca.show(truncate=False)

## Conclusión pregunta 2

La marca con mayor límite de crédito promedio es **Visa**, con un límite promedio de **14,737.33**.

Le sigue **Mastercard**, con **14,659.60**, y después **Amex**, con **11,436.32**.

Aunque Mastercard tiene más tarjetas registradas, Visa presenta el mayor límite promedio. Esto puede indicar que Visa está asociada a productos o segmentos con mayor capacidad crediticia dentro de este dataset.

In [0]:
# Pregunta 3: ¿Las tarjetas con chip tienen límites de crédito diferentes?

df_chip_limite = (
    df_clean
    .groupBy("has_chip")
    .agg(
        F.count("*").alias("total_tarjetas"),
        F.round(F.avg("credit_limit_num"), 2).alias("limite_promedio"),
        F.percentile_approx("credit_limit_num", 0.5).alias("mediana_limite"),
        F.max("credit_limit_num").alias("limite_maximo")
    )
    .orderBy(F.col("limite_promedio").desc())
)

df_chip_limite.show(truncate=False)

## Conclusión pregunta 3

Las tarjetas con chip tienen un límite promedio de **14,353.82**, mientras que las tarjetas sin chip tienen un límite promedio de **14,293.61**.

La diferencia es pequeña, por lo que no parece existir una separación fuerte entre tarjetas con chip y sin chip en términos de límite promedio.

Sin embargo, las tarjetas con chip son mucho más frecuentes: **5,500 tarjetas** frente a **646 tarjetas** sin chip. Esto sugiere que la tecnología chip está ampliamente adoptada en el portafolio.

## Parte 6 — Reflexión final

### ¿Qué fue lo más sorprendente que encontraste en los datos?

Lo más sorprendente fue que la mayoría de tarjetas del dataset son de tipo **Debit**, representando más de la mitad del portafolio. También llamó la atención que las tarjetas con chip son mucho más frecuentes que las tarjetas sin chip.

### ¿Qué transformación te resultó más difícil de aplicar a tu dataset y por qué?

La transformación más importante fue limpiar `credit_limit` y convertirlo a número. Fue necesaria porque los valores financieros suelen venir como texto con símbolos o separadores, y para calcular promedios o máximos era necesario convertirlos a tipo numérico.

### ¿Qué pregunta de negocio adicional te gustaría responder si tuvieras más tiempo?

Me gustaría cruzar este dataset con las transacciones para analizar si ciertos tipos de tarjeta, marcas o límites de crédito están asociados con mayor fraude o mayor volumen transaccional.

### ¿En qué se diferencia explorar este dataset con PySpark respecto a hacerlo con pandas o Excel?

Con PySpark se trabaja pensando en procesamiento distribuido y transformaciones lazy. A diferencia de pandas o Excel, Spark no ejecuta todo inmediatamente, sino que construye un plan de ejecución y lo materializa cuando se llama una acción como `show()`, `count()` o `write()`.

Esto hace que PySpark sea más adecuado para datasets grandes y pipelines productivos, aunque requiere ser más cuidadoso con los tipos de datos, las acciones y el costo de las operaciones.